# 1. Test sets

The paper evaluates on three sets built from the validation split of FOLIO, a collection of short stories written
by experts, each with a conclusion labelled True, False or Uncertain and a first-order formula for every sentence.

* **FOLIO original** keeps the stories whose expert formulas reproduce the label under Z3.
* **FOLIO with new names** gives every named individual a new name and shuffles the premises. The label does not change.
* **FOLIO counterfactual** also removes one premise and recomputes the label with Z3. A system that remembers the
  original story approves claims that no longer follow.

The sets are built by fixed rules: no sentence or label is written by a person or a model.

Runs on a CPU in about half a minute and needs network access to download FOLIO.

In [1]:
# Copyright 2026 Javier Marín
# SPDX-License-Identifier: Apache-2.0

from pathlib import Path

REPO = Path.cwd().parent
DATA = REPO / "data"
RUNS = REPO / "runs"      # everything this notebook produces goes here; data/ is never overwritten

import collections

from logicverifier import data

## Download FOLIO

The validation split is read at a fixed commit of the FOLIO repository.

In [2]:
validation = data.read_jsonl(data.download_folio("validation", RUNS / "folio" / "folio-validation.jsonl"))
len(validation)

204

## Build the sets

Every story is checked with Z3 at each step. Formulas are normalised before their vocabulary is extracted, so that
names such as `Mammal (x)` or `LostToIgaŚwiątek` reach the translator in a form the grammar accepts.

In [3]:
sets, report = data.build_test_sets(validation, seed=0)
report

{'folio_original': 179,
 'folio_new_names': 179,
 'folio_counterfactual': 443,
 'counterfactual_labels': {'Uncertain': 360, 'True': 47, 'False': 36},
 'stories_used': 179,
 'drop_control': 147,
 'drop_control:Uncertain->Uncertain': 64,
 'drop_changed': 296,
 'drop_changed:True->Uncertain': 159,
 'drop_control:True->True': 47,
 'skipped_formulas_disagree_with_label': 15,
 'skipped_shape': 10,
 'drop_changed:False->Uncertain': 137,
 'drop_control:False->False': 36}

## Compare with the sets used in the paper

In [4]:
for name, stories in sets.items():
    data.write_jsonl(stories, RUNS / "test_sets" / f"{name}.jsonl")
    stored = data.read_jsonl(DATA / "test_sets" / f"{name}.jsonl")
    print(f"{data.SETS[name]:24s} {len(stories):4d} stories   {'identical' if stories == stored else 'DIFFERENT'}")

FOLIO original            179 stories   identical
FOLIO with new names      179 stories   identical
FOLIO counterfactual      443 stories   identical


In [5]:
labels = collections.Counter(story["label"] for stories in sets.values() for story in stories)
print(f"{sum(labels.values())} stories: {labels['True']} True, {labels['False']} False, {labels['Uncertain']} Uncertain")

801 stories: 171 True, 142 False, 488 Uncertain
